# 6 · Non-redundant K-fold (NR-Kfold) analysis

1. Embed train + val + test with each pretrained backbone, k-means each class into 5 clusters, and assign whole clusters to 5 folds so test folds are dissimilar from the training folds.
2. Train a linear probe on 4 folds and test on the held-out fold, for every fold.
3. Summarise per dataset and rank across datasets (dissertation Table 11).

**Writes:** `Stats/<dataset>/<model>/distribution_shift/` (embeddings, clusters, folds, t-SNE, probe metrics), `Stats/<dataset>_kfold_summary.xlsx`, `Stats/aggregate_kfold_model_rankings.xlsx`, `Stats/per_fold_visuals/`.

H0-mini was excluded from this analysis in the dissertation.

Paths are resolved automatically (repo root = folder with `pyproject.toml`). If `Data/`, `Models/` and `Stats/` live elsewhere (e.g. on the cluster), set `PFM_STORAGE=/path/to/storage` before starting Jupyter. Gated Hugging Face models need `HF_TOKEN` set.

In [ ]:
from pfm_robustness import DATASETS, MODELS
from pfm_robustness.kfold import build_folds, summarise_kfold
from pfm_robustness.plots import (load_kfold_workbook, plot_fold_boxplot, plot_fold_heatmap,
                                  plot_kfold_vs_params)

## Configure

In [ ]:
DATASET_NAMES = list(DATASETS)
MODEL_NAMES = [m for m in MODELS if m != "H0-mini"]

PROBE = dict(epochs=25, batch_size=64, lr=1e-3, weight_decay=1e-4, val_fraction=0.3, patience=5)

## 1. Embeddings, clusters and folds

In [ ]:
build_folds(DATASET_NAMES, MODEL_NAMES)

## 2–3. Linear probe per fold and summaries

In [ ]:
ranking = summarise_kfold(DATASET_NAMES, MODEL_NAMES, **PROBE)
ranking

## Figures

In [ ]:
for ds in DATASET_NAMES:
    tables = load_kfold_workbook(ds)
    plot_fold_heatmap(ds, tables)
    plot_fold_boxplot(ds, tables)

plot_kfold_vs_params()